# Padding et collate_fn : assembler des durées variables

Construire un rectangle tout en conservant la longueur de chaque extrait.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#padding-et-collate).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Notre Dataset simple fonctionne parce que toutes ses lignes ont la même taille. Nous changeons maintenant d’expérience : chaque extrait devient une suite de mesures au cours du temps.

La voix a deux instants, la percussion un seul. À chaque instant, nous gardons deux descripteurs fictifs : RMS et facteur de crête. Ces valeurs sont encore choisies à la main.

| Extrait | Instant 0 : [RMS,crête] | Instant 1 : [RMS,crête] | Cible |
| --- | --- | --- | --- |
| Voix | [0,1 ; 1,5] | [0,2 ; 2,5] | 0 |
| Percussion | [0,2 ; 4,5] | Absent | 1 |

Chaque matrice est (temps,descripteurs). Une ligne nulle sera ajoutée à la percussion pour obtenir (extraits,temps,descripteurs) = (2,2,2).

La fonction collate_fn reçoit une liste d’exemples et construit un lot. Nous lui demandons aussi les longueurs et un masque : vrai pour une position observée, faux pour une case ajoutée.

## Démonstration 1 · Dépaqueter, compléter, puis marquer les positions

Suivre les deux exemples sans confondre les zéros ajoutés et les vrais zéros.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader


### Calculer l’état suivant

exemples contient deux couples : (matrice de voix,0) et (matrice de percussion,1). Les cibles ne sont pas des colonnes du signal.

In [4]:
voix = torch.tensor([[0.1, 1.5], [0.2, 2.5]])
percussion = torch.tensor([[0.2, 4.5]])
exemples = [(voix, 0), (percussion, 1)]



In [5]:
observer('voix', voix)
observer('percussion', percussion)
observer('exemples', exemples)

voix forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 2.5]]
percussion forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.20000000298023224, 4.5]]
exemples : 2 éléments ; formes des tenseurs : []


### Définir la règle avant de l’utiliser

L’étoile donne les deux couples comme arguments à zip. zip les regroupe par position : les matrices ensemble, puis les deux cibles.

Pour [(A,0),(B,1)], zip(*exemples) produit (A,B), puis (0,1). Nous donnons un nom à chacun de ces deux groupes.

len(signal) compte les lignes temporelles. La compréhension calcule donc [2,1], puis torch.tensor en fait un tenseur.

pad_sequence complète jusqu’à la plus grande longueur. batch_first=True met les extraits en premier ; padding_value=0 choisit la valeur ajoutée.

arange(2) produit [0,1]. None ajoute un axe de taille un : les positions deviennent une ligne et les longueurs une colonne.

Le broadcasting aligne la même ligne [0,1] sur les deux extraits. Pour une longueur 1, seule la position 0 est strictement inférieure à 1.

Nous reconstruisons une annotation entière par extrait. Le masque a les axes (extrait,temps), sans axe des descripteurs.

In [6]:
def assembler(exemples):
    signaux, cibles = zip(*exemples)
    longueurs = torch.tensor([len(signal) for signal in signaux])
    valeurs = pad_sequence(signaux, batch_first=True, padding_value=0.)
    positions = torch.arange(valeurs.shape[1])
    positions_ligne = positions[None, :]
    longueurs_colonne = longueurs[:, None]
    masque = positions_ligne < longueurs_colonne
    annotations = torch.tensor(cibles, dtype=torch.long)
    return valeurs, annotations, longueurs, masque



### Calculer l’état suivant

L’appel direct aide à comprendre la fonction. Ensuite DataLoader l’appelle lui-même pour chaque liste d’exemples.

In [7]:
batch, cibles, longueurs, masque = assembler(exemples)


In [8]:
observer('batch', batch)
observer('cibles', cibles)
observer('longueurs', longueurs)
observer('masque', masque)

batch forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.10000000149011612, 1.5], [0.20000000298023224, 2.5]], [[0.20000000298023224, 4.5], [0.0, 0.0]]]
cibles forme (2,) type torch.int64 requires_grad False
  valeurs : [0, 1]
longueurs forme (2,) type torch.int64 requires_grad False
  valeurs : [2, 1]
masque forme (2, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, False]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [9]:
print("lot :", batch)
print("formes :", tuple(batch.shape), tuple(masque.shape))
print("longueurs :", longueurs.tolist(), "cibles :", cibles.tolist())
print("masque :", masque.tolist())


lot : tensor([[[0.1000, 1.5000],
         [0.2000, 2.5000]],

        [[0.2000, 4.5000],
         [0.0000, 0.0000]]])
formes : (2, 2, 2) (2, 2)
longueurs : [2, 1] cibles : [0, 1]
masque : [[True, True], [True, False]]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
loader = DataLoader(exemples, batch_size=2, collate_fn=assembler, shuffle=False)
lot_du_loader = next(iter(loader))


In [11]:
observer('lot_du_loader', lot_du_loader)

lot_du_loader = (tensor([[[0.1000, 1.5000],
         [0.2000, 2.5000]],

        [[0.2000, 4.5000],
         [0.0000, 0.0000]]]), tensor([0, 1]), tensor([2, 1]), tensor([[ True,  True],
        [ True, False]]))


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [12]:
print("meme assemblage :", torch.equal(lot_du_loader[0], batch))

meme assemblage : True


Le lot contient les valeurs ; les longueurs et le masque indiquent lesquelles étaient réellement présentes.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Deux séquences contiennent deux positions puis une seule. Complétez leur masque de validité après padding.

In [13]:
exo_longueurs = torch.tensor([2,1])
exo_positions = torch.arange(2)[None,:]
exo_masque = ...  # Remplacer les points par votre réponse.
if exo_masque is Ellipsis or exo_masque is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_masque),exo_positions<exo_longueurs[:,None])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La deuxième case du second extrait a été ajoutée pour le stockage.

</details>

### Exercice 2

Une séquence [5] est stockée comme [5,9]. Le second nombre est du padding. Quelle moyenne pertinente conserve-t-on ?

In [14]:
exo_x = torch.tensor([5.,9.])
exo_valide = torch.tensor([True,False])
exo_moyenne = ...  # Remplacer les points par votre réponse.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le masque dépend de la longueur d’origine, pas de la valeur zéro.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_longueurs = torch.tensor([2,1])
exo_positions = torch.arange(2)[None,:]
exo_masque = torch.tensor([[True,True],[True,False]])
if exo_masque is Ellipsis or exo_masque is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_masque),exo_positions<exo_longueurs[:,None])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([5.,9.])
exo_valide = torch.tensor([True,False])
exo_moyenne = 5.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [15]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()